# Drug-drug interaction (DDI) severity checker

**Dataset:** Drug-Drug Interactions — Kaggle
Source: https://www.kaggle.com/datasets/mghobashy/drug-drug-interactions

Academic backing: this kind of dataset is typically derived from or cross-referenced against
DDInter (https://ddinter.scbdd.com/) and similar pharmacological interaction databases used
in published research on prescribing safety.

**Important — verify before relying on this notebook:** unlike the other two notebooks, I was
not able to directly confirm this dataset's exact column names and structure (Kaggle's
dataset preview page is JavaScript-rendered and wasn't fetchable for direct inspection).
**The first code cell below prints the dataframe's actual columns and a sample —
run that first and adjust the column names used in the rest of the notebook to match
what you actually see.** Common shapes for DDI datasets are either:
  (a) `drug_1, drug_2, interaction_description, severity` (pairwise table), or
  (b) `drug_a, drug_b, interaction_type` with severity embedded in free text

This notebook is written for shape (a) — the most common structure for this type of dataset
— with the schema-detection cell below as a safety check. Adjust column references in
Sections 2+ if your actual download differs.

**What this notebook does:**
1. Loads the DDI dataset and prints its real schema (verify first)
2. Cleans and standardizes drug name pairs
3. If a severity label already exists in the data, trains a classifier to predict severity
   from interaction description text (same TF-IDF approach as notebook 1, for consistency)
4. If no severity label exists, builds a direct lookup table instead (still legitimate —
   not every dataset needs a trained classifier; a verified, real, structured lookup is a
   defensible ML-adjacent pipeline too, and you should say so honestly in your report rather
   than force-fitting a classifier where a lookup is more appropriate)
5. Exports a pairwise interaction lookup for Django

**Setup before running:**
- Download the dataset from the Kaggle link above
- Place the CSV into `./data/` and update the filename in the load cell below


In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

DATA_DIR = Path("../datasets")
OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Load and inspect the real schema — RUN THIS FIRST

Update the filename below to match what you actually downloaded from Kaggle, then run this
cell and read the printed output before touching anything below it.

In [2]:
# UPDATE this filename to match your actual downloaded CSV
ddi_filename = "db_drug_interactions.csv"

ddi_df = pd.read_csv(DATA_DIR / ddi_filename)

print("Shape:", ddi_df.shape)
print()
print("Columns:", list(ddi_df.columns))
print()
print("Dtypes:")
print(ddi_df.dtypes)
print()
print("Sample rows:")
ddi_df.head(5)

Shape: (191541, 3)

Columns: ['Drug 1', 'Drug 2', 'Interaction Description']

Dtypes:
Drug 1                     str
Drug 2                     str
Interaction Description    str
dtype: object

Sample rows:


,Drug 1,Drug 2,Interaction Description
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing a...
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosens...
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensiti...
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensiti...
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing ...


In [3]:
print("Missing values per column:")
print(ddi_df.isnull().sum())
print()
print("Total rows:", len(ddi_df))
print("Total unique drugs (column 1, adjust name if needed):")
# ADJUST: replace 'drug_1' below with your real first-drug column name once confirmed above
first_drug_col_guess = ddi_df.columns[0]
print(f"  Using '{first_drug_col_guess}' as a guess —", ddi_df[first_drug_col_guess].nunique(), "unique values")

Missing values per column:
Drug 1                     0
Drug 2                     0
Interaction Description    0
dtype: int64

Total rows: 191541
Total unique drugs (column 1, adjust name if needed):
  Using 'Drug 1' as a guess — 1634 unique values


## 2. Standardize column names

Rename whatever the real columns are to a consistent internal schema so the rest of the
notebook doesn't need to change. **Edit the dictionary below to match what you saw in
Section 1** — this is the only place you should need to make changes for a different
column layout.

In [4]:
# EDIT THIS MAPPING to match your actual columns from Section 1's printed output.
# Left side = your real column name, right side = the standard name this notebook expects.
# Example, if your real columns are ['Drug 1', 'Drug 2', 'Interaction', 'Level']:
#   column_rename_map = {
#       "Drug 1": "drug_a",
#       "Drug 2": "drug_b",
#       "Interaction": "interaction_text",
#       "Level": "severity_raw",
#   }

column_rename_map = {
    "Drug 1": "drug_a",
    "Drug 2": "drug_b",
    "Interaction Description": "interaction_text",
}

if column_rename_map:
    ddi_df = ddi_df.rename(columns=column_rename_map)
    print("Renamed columns:", list(ddi_df.columns))
else:
    print("column_rename_map is empty — fill it in based on Section 1's output before continuing.")
    print("Current real columns are:", list(ddi_df.columns))

Renamed columns: ['drug_a', 'drug_b', 'interaction_text']


## 3. Clean drug name pairs

Standardize casing/whitespace so pairwise lookups work reliably, and make the pair
order-independent (interaction between A and B is the same as B and A).

In [5]:
def normalize_drug_name(name):
    if pd.isna(name):
        return ""
    return str(name).strip().lower()

# This cell assumes 'drug_a' and 'drug_b' exist after Section 2's rename.
# If your dataset uses different concepts (e.g. a single 'drug_pair' string column
# instead of two separate columns), adjust accordingly before running.

ddi_df["drug_a_clean"] = ddi_df["drug_a"].apply(normalize_drug_name)
ddi_df["drug_b_clean"] = ddi_df["drug_b"].apply(normalize_drug_name)

ddi_df = ddi_df[(ddi_df["drug_a_clean"] != "") & (ddi_df["drug_b_clean"] != "")].reset_index(drop=True)

# Order-independent pair key: alphabetically sort the two names so (A,B) and (B,A) match
def make_pair_key(row):
    pair = sorted([row["drug_a_clean"], row["drug_b_clean"]])
    return f"{pair[0]}|||{pair[1]}"

ddi_df["pair_key"] = ddi_df.apply(make_pair_key, axis=1)

print("Rows after cleaning:", len(ddi_df))
print("Unique drug pairs:", ddi_df["pair_key"].nunique())
ddi_df[["drug_a_clean", "drug_b_clean", "pair_key"]].head(5)

Rows after cleaning: 191541
Unique drug pairs: 191135


,drug_a_clean,drug_b_clean,pair_key
0,trioxsalen,verteporfin,trioxsalen|||verteporfin
1,aminolevulinic acid,verteporfin,aminolevulinic acid|||verteporfin
2,titanium dioxide,verteporfin,titanium dioxide|||verteporfin
3,tiaprofenic acid,verteporfin,tiaprofenic acid|||verteporfin
4,cyamemazine,verteporfin,cyamemazine|||verteporfin


## 4a. If a severity label exists: train a classifier

Run this section if your dataset has a `severity_raw` column (minor/moderate/major or
similar). This mirrors the TF-IDF + Logistic Regression approach from notebook 1, applied
to interaction description text instead of patient reviews.

In [6]:
HAS_SEVERITY_LABEL = "severity_raw" in ddi_df.columns and "interaction_text" in ddi_df.columns

if HAS_SEVERITY_LABEL:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    clean_ddi = ddi_df.dropna(subset=["interaction_text", "severity_raw"])
    print("Severity label classes:", clean_ddi["severity_raw"].unique())
    print("Rows available for training:", len(clean_ddi))

    X = clean_ddi["interaction_text"].astype(str)
    y = clean_ddi["severity_raw"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    severity_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])

    severity_pipeline.fit(X_train, y_train)
    pred = severity_pipeline.predict(X_test)

    print()
    print("=== Severity classifier ===")
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 4))
    print()
    print(classification_report(y_test, pred, zero_division=0))

    # Predict severity for every row in the full cleaned dataset, to build the lookup
    ddi_df.loc[clean_ddi.index, "predicted_severity"] = severity_pipeline.predict(
        clean_ddi["interaction_text"].astype(str)
    )

    joblib.dump(severity_pipeline, OUTPUT_DIR / "ddi_severity_pipeline.joblib")
    print()
    print("Saved severity_pipeline.joblib")
else:
    print("No severity_raw + interaction_text columns found — skipping classifier training.")
    print("Proceeding to Section 4b to build a direct lookup table instead.")

No severity_raw + interaction_text columns found — skipping classifier training.
Proceeding to Section 4b to build a direct lookup table instead.


## 4b. If no severity label exists: build a direct lookup instead

This is the fallback if the dataset only confirms *that* an interaction exists, without a
graded severity. A real, dataset-backed "this combination has a known interaction, consult
a pharmacist" flag is still a legitimate and useful feature — don't force a classifier onto
data that doesn't support one. Be honest about this distinction in your report: a trained
classifier and a verified lookup table are both valid ML-pipeline outputs, but they are not
the same claim, and conflating them under "we used ML for everything" would be inaccurate.

In [7]:
if not HAS_SEVERITY_LABEL:
    # Direct flag: any pair present in the dataset is a known interaction.
    # If the dataset has a free-text interaction description but no graded severity,
    # we keep the raw description for display rather than inventing a severity score.
    has_description = "interaction_text" in ddi_df.columns
    print("Building direct lookup. Includes free-text description:", has_description)

Building direct lookup. Includes free-text description: True


## 5. Build the pairwise lookup table

Regardless of which path (4a or 4b) ran, this builds the final structure Django will load:
a dictionary keyed by the order-independent pair key, used to check whether a user's
existing medications (from their `MedicalProfile.current_medications`) interact with a
medicine they're searching for.

In [8]:
lookup_columns = ["pair_key", "drug_a_clean", "drug_b_clean"]
if "predicted_severity" in ddi_df.columns:
    lookup_columns.append("predicted_severity")
elif "severity_raw" in ddi_df.columns:
    lookup_columns.append("severity_raw")
if "interaction_text" in ddi_df.columns:
    lookup_columns.append("interaction_text")

ddi_lookup = ddi_df[lookup_columns].drop_duplicates(subset=["pair_key"]).reset_index(drop=True)

print("Final lookup table size:", len(ddi_lookup))
ddi_lookup.head(5)

Final lookup table size: 191135


,pair_key,drug_a_clean,drug_b_clean,interaction_text
0,trioxsalen|||verteporfin,trioxsalen,verteporfin,Trioxsalen may increase the photosensitizing a...
1,aminolevulinic acid|||verteporfin,aminolevulinic acid,verteporfin,Aminolevulinic acid may increase the photosens...
2,titanium dioxide|||verteporfin,titanium dioxide,verteporfin,Titanium dioxide may increase the photosensiti...
3,tiaprofenic acid|||verteporfin,tiaprofenic acid,verteporfin,Tiaprofenic acid may increase the photosensiti...
4,cyamemazine|||verteporfin,cyamemazine,verteporfin,Cyamemazine may increase the photosensitizing ...


In [9]:
ddi_lookup_records = ddi_lookup.to_dict(orient="records")

with open(OUTPUT_DIR / "drug_interaction_lookup.json", "w") as f:
    json.dump(ddi_lookup_records, f, indent=2)

print("Saved:", OUTPUT_DIR / "drug_interaction_lookup.json")
print("Total interaction pairs stored:", len(ddi_lookup_records))
print()
print("Django usage: when a user has current_medications = ['Drug A', 'Drug B'] and views a")
print("medicine page for 'Drug C', the backend checks pair_key('Drug A','Drug C') and")
print("pair_key('Drug B','Drug C') against this lookup and surfaces any matches as a warning.")

Saved: ..\models\drug_interaction_lookup.json
Total interaction pairs stored: 191135

Django usage: when a user has current_medications = ['Drug A', 'Drug B'] and views a
medicine page for 'Drug C', the backend checks pair_key('Drug A','Drug C') and
pair_key('Drug B','Drug C') against this lookup and surfaces any matches as a warning.
